# 🔧 Session 2+3: Naive RAG — Einrichtung & Chunking

**PIRAGE Trainingskurs — Tag 1, Session 2 & 3**

> *Kein API-Key erforderlich — läuft vollständig im Browser via Google Colab*

---

## 🎯 Lernziele

- Den vollständigen RAG-Workflow verstehen: **Parse → Index → Retrieval → Augmented Generation**
- **Chunking-Strategien** kennen und vergleichen (fest vs. semantisch)
- Eine vollständige Naive RAG-Pipeline von Grund auf aufbauen
- Warum Chunking die Qualität des Retrievals entscheidend beeinflusst

---

## 📦 Setup

Wir verwenden:
- `sentence-transformers` — kostenlose, lokale Embeddings
- `faiss-cpu` — schnelle Vektorsuche
- `rank_bm25` — klassisches Textsuch-Ranking (für später)

In [ ]:
%pip install -q sentence-transformers faiss-cpu rank_bm25 numpy

In [ ]:
import re
import time
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

print('✅ Pakete geladen')

## 📄 Teil 1 — Parse (P in PIRAGE)

### Der RAG-Workflow beginnt mit Dokumenten

In der Praxis liest man PDFs, Word-Dateien, HTML-Seiten usw. ein.
Hier simulieren wir das mit Beispieldokumenten in Python-Strings.

> In echten Projekten würde man z.B. `PyMuPDF`, `python-docx` oder `unstructured` nutzen.

In [ ]:
# Simuliertes Unternehmenshandbuch — in der Praxis aus PDFs/Datenbanken
dokumente = [
    {
        "quelle": "handbuch_hr.txt",
        "inhalt": """
        Urlaubsanspruch und Abwesenheitsregelungen

        Vollzeitbeschäftigte haben Anspruch auf 30 Arbeitstage Urlaub pro Kalenderjahr.
        Teilzeitbeschäftigte erhalten den Urlaub anteilig berechnet.

        Die Beantragung von Urlaub muss mindestens zwei Wochen im Voraus über das
        HR-System erfolgen und von der jeweiligen Führungskraft genehmigt werden.
        Urlaubssperren werden rechtzeitig bekannt gegeben.

        Resturlaub aus dem aktuellen Kalenderjahr kann bis zum 31. März des Folgejahres
        genommen werden. Danach verfällt er ersatzlos. Eine Auszahlung von Urlaub ist
        nur bei Beendigung des Arbeitsverhältnisses möglich.

        Für Krankheitsfälle gilt: Nach drei Fehltagen ist eine ärztliche
        Krankmeldung (AU-Bescheinigung) einzureichen. Bei längerer Krankheit greift
        die Lohnfortzahlung für bis zu sechs Wochen.
        """
    },
    {
        "quelle": "handbuch_it.txt",
        "inhalt": """
        IT-Sicherheitsrichtlinien

        Passwortrichtlinien:
        Alle Passwörter müssen mindestens 12 Zeichen lang sein und Groß-/Kleinbuchstaben,
        Zahlen sowie Sonderzeichen enthalten. Passwörter sind alle 90 Tage zu erneuern.
        Die Verwendung von Passwort-Managern (z.B. Bitwarden, 1Password) wird empfohlen.
        Passwörter dürfen nicht geteilt oder schriftlich notiert werden.

        Remote-Arbeit und VPN:
        Bei der Arbeit von externen Standorten muss das Unternehmens-VPN aktiv sein.
        Der VPN-Client wird von der IT-Abteilung bereitgestellt und muss vor der
        ersten Nutzung eingerichtet werden. Bei Problemen: it-support@firma.de

        Externe Speichermedien:
        Die Verwendung privater USB-Sticks und externer Festplatten ist untersagt.
        Für Dateiübertragungen ist ausschließlich die Unternehmens-Cloud zu nutzen.
        """
    },
    {
        "quelle": "handbuch_reise.txt",
        "inhalt": """
        Reisekostenregelung und Dienstreisen

        Genehmigung:
        Dienstreisen müssen vor Antritt durch die Führungskraft genehmigt werden.
        Die Buchung von Bahn und Flug erfolgt über das Travel-Portal oder das
        Reisebüro der Firma. Privatbuchungen werden nur in Ausnahmefällen erstattet.

        Unterkünfte:
        Das maximale Hotelbudget beträgt 120 Euro pro Nacht inkl. Frühstück.
        In Städten mit höheren Übernachtungspreisen (München, Frankfurt, Hamburg)
        gilt ein erhöhtes Budget von 150 Euro. Die Buchung muss über das Portal erfolgen.

        Verpflegungspauschalen:
        Bei Dienstreisen ab 8 Stunden: 14 Euro (Inland).
        Bei Dienstreisen ab 24 Stunden: 28 Euro (Inland).
        Auslandsreisen: Länderspezifische Pauschalen gemäß Bundesreisekostengesetz.

        Abrechnung:
        Alle Belege müssen innerhalb von 4 Wochen nach der Reise eingereicht werden.
        Erstattung erfolgt mit der nächsten Gehaltsabrechnung.
        """
    }
]

print(f'📚 {len(dokumente)} Dokumente geladen')
for d in dokumente:
    worte = len(d['inhalt'].split())
    print(f'  📄 {d["quelle"]} — {worte} Wörter')

## ✂️ Teil 2 — Chunking-Strategien

### Warum Chunking so wichtig ist

Embedding-Modelle haben eine **maximale Eingabelänge** (meist 512 Token).
Lange Dokumente müssen aufgeteilt werden. Aber wie?

```
Ganzes Dokument (2000 Wörter)
       ↓
┌──────┬──────┬──────┬──────┐
│Chunk1│Chunk2│Chunk3│Chunk4│  ← fest (fixed-size)
└──────┴──────┴──────┴──────┘

       ↓ besser:

┌────────────┬────────────┬──────────────┐
│  Abschnitt │  Abschnitt │   Abschnitt  │  ← semantisch (nach Absätzen)
│  1 & 2     │    3       │   4 & 5      │
└────────────┴────────────┴──────────────┘
```

### Strategie 1: Feste Chunk-Größe

In [ ]:
def chunking_fest(text: str, chunk_groesse: int = 200, ueberlappung: int = 50) -> list[str]:
    """
    Teilt Text in Chunks fester Wortanzahl auf.
    Überlappung verhindert, dass Satzgrenzen wichtige Info abtrennen.
    """
    woerter = text.split()
    chunks = []
    i = 0
    while i < len(woerter):
        chunk = ' '.join(woerter[i:i + chunk_groesse])
        if chunk.strip():
            chunks.append(chunk)
        i += chunk_groesse - ueberlappung
    return chunks


# Test
test_doc = dokumente[0]['inhalt']
chunks_fest = chunking_fest(test_doc, chunk_groesse=80, ueberlappung=20)

print(f'📊 Festes Chunking (80 Wörter, 20 Überlappung):')
print(f'   Dokument: {len(test_doc.split())} Wörter → {len(chunks_fest)} Chunks\n')
for i, chunk in enumerate(chunks_fest):
    print(f'[Chunk {i+1}] {chunk[:120]}...')
    print()

### Strategie 2: Semantisches Chunking (nach Absätzen)

In [ ]:
def chunking_semantisch(text: str, min_woerter: int = 30) -> list[str]:
    """
    Teilt Text an natürlichen Absatzgrenzen auf.
    Kurze Absätze werden zusammengeführt.
    """
    absaetze = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

    chunks = []
    aktuell = ''

    for absatz in absaetze:
        if len(aktuell.split()) + len(absatz.split()) < min_woerter * 3:
            aktuell = (aktuell + ' ' + absatz).strip()
        else:
            if aktuell:
                chunks.append(aktuell)
            aktuell = absatz

    if aktuell:
        chunks.append(aktuell)

    return chunks


chunks_semantisch = chunking_semantisch(test_doc)

print(f'📊 Semantisches Chunking (nach Absätzen):')
print(f'   Dokument: {len(test_doc.split())} Wörter → {len(chunks_semantisch)} Chunks\n')
for i, chunk in enumerate(chunks_semantisch):
    print(f'[Chunk {i+1}] ({len(chunk.split())} Wörter)')
    print(f'  {chunk[:150]}...')
    print()

### 🔬 Vergleich: Welche Strategie ist besser?

Wir bauen gleich zwei Indizes und vergleichen die Retrieval-Qualität.

In [ ]:
# Alle Dokumente chunken
def alle_chunks_erstellen(dokumente: list, strategie: str = 'semantisch') -> list[dict]:
    """Verarbeitet alle Dokumente und gibt eine Liste von Chunks mit Metadaten zurück."""
    alle_chunks = []
    for doc in dokumente:
        if strategie == 'fest':
            chunks = chunking_fest(doc['inhalt'], chunk_groesse=80, ueberlappung=20)
        else:
            chunks = chunking_semantisch(doc['inhalt'])

        for i, chunk in enumerate(chunks):
            alle_chunks.append({
                'text': chunk,
                'quelle': doc['quelle'],
                'chunk_nr': i
            })

    return alle_chunks


chunks_fest_alle = alle_chunks_erstellen(dokumente, 'fest')
chunks_sem_alle = alle_chunks_erstellen(dokumente, 'semantisch')

print(f'📊 Vergleich Chunking-Strategien:')
print(f'   Festes Chunking:      {len(chunks_fest_alle)} Chunks')
print(f'   Semantisches Chunking: {len(chunks_sem_alle)} Chunks')

## 🗂️ Teil 3 — Index aufbauen (I in PIRAGE)

In [ ]:
print('🔄 Lade Embedding-Modell...')
modell = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Modell bereit\n')


def index_aufbauen(chunks: list[dict]) -> tuple:
    """Erstellt FAISS-Index aus einer Liste von Chunks."""
    texte = [c['text'] for c in chunks]

    print(f'🔄 Erstelle Embeddings für {len(texte)} Chunks...')
    start = time.time()
    embeddings = modell.encode(texte, show_progress_bar=True, batch_size=32)
    dauer = time.time() - start
    print(f'✅ Embeddings in {dauer:.1f}s erstellt')

    dim = embeddings.shape[1]
    idx = faiss.IndexFlatL2(dim)
    idx.add(embeddings.astype('float32'))
    print(f'✅ FAISS-Index mit {idx.ntotal} Vektoren ({dim} Dimensionen)')

    return idx, embeddings


print('--- Index mit semantischem Chunking ---')
index_sem, emb_sem = index_aufbauen(chunks_sem_alle)

print('\n--- Index mit festem Chunking ---')
index_fest, emb_fest = index_aufbauen(chunks_fest_alle)

## 🔍 Teil 4 — Retrieval & Vergleich (R in PIRAGE)

In [ ]:
def retrieval(frage: str, idx: faiss.Index, chunks: list[dict], k: int = 3) -> list[dict]:
    """Findet die k relevantesten Chunks für eine Frage."""
    frage_emb = modell.encode([frage]).astype('float32')
    distanzen, indizes = idx.search(frage_emb, k)

    ergebnisse = []
    for abstand, chunk_idx in zip(distanzen[0], indizes[0]):
        ergebnisse.append({
            **chunks[chunk_idx],
            'score': float(1 / (1 + abstand))
        })
    return ergebnisse


def vergleich_chunking(frage: str):
    """Zeigt den Unterschied zwischen fester und semantischer Chunking-Strategie."""
    print(f'❓ Frage: "{frage}"')
    print('=' * 70)

    for name, idx, chunks in [
        ('SEMANTISCHES Chunking', index_sem, chunks_sem_alle),
        ('FESTES Chunking', index_fest, chunks_fest_alle)
    ]:
        print(f'\n📌 {name}:')
        ergebnisse = retrieval(frage, idx, chunks, k=2)
        for i, r in enumerate(ergebnisse):
            print(f'  [{i+1}] Score: {r["score"]:.3f} | Quelle: {r["quelle"]}')
            print(f'       {r["text"][:200]}...')
    print()


# Testfragen
testfragen = [
    "Wann verfällt mein Resturlaub?",
    "Wie hoch ist das Hotelbudget für Dienstreisen?",
    "Was mache ich wenn ich krank bin?"
]

for frage in testfragen:
    vergleich_chunking(frage)

## 🤖 Teil 5 — Augmented Generation (AG in PIRAGE)

### Prompt-Konstruktion ohne echtes LLM

Hier zeigen wir, wie der finale Prompt aufgebaut wird.
In der Praxis würde dieser Prompt an ein LLM (lokal oder API) gesendet.

> **Lokale LLM-Optionen:** Ollama (llama3, mistral), LM Studio, llamafile
> **Cloud-Optionen:** OpenAI API, Azure OpenAI, Anthropic Claude

In [ ]:
def prompt_erstellen(frage: str, kontext_chunks: list[dict]) -> str:
    """Erstellt den vollständigen RAG-Prompt für ein LLM."""
    kontext_text = '\n\n---\n\n'.join([
        f'Quelle: {c["quelle"]}\n{c["text"]}'
        for c in kontext_chunks
    ])

    prompt = f"""Du bist ein hilfreicher Assistent für Unternehmensfragen.
Beantworte die Frage ausschließlich auf Basis des folgenden Kontexts.
Falls die Information nicht im Kontext vorhanden ist, sage das ehrlich.

=== KONTEXT ===
{kontext_text}

=== FRAGE ===
{frage}

=== ANTWORT ==="""
    return prompt


# Demo: Zeige den vollständigen Prompt
beispiel_frage = "Wie viele Urlaubstage habe ich und was passiert mit Resturlaub?"
kontext = retrieval(beispiel_frage, index_sem, chunks_sem_alle, k=3)

prompt = prompt_erstellen(beispiel_frage, kontext)

print('📝 Vollständiger RAG-Prompt:')
print('─' * 70)
print(prompt)
print('─' * 70)
print(f'\n📊 Prompt-Länge: {len(prompt.split())} Wörter / ~{len(prompt)} Zeichen')
print('\n💡 Dieser Prompt wird an ein LLM gesendet → präzise, faktenbasierte Antwort!')

In [ ]:
# Vollständige naive_rag Funktion
def naive_rag(frage: str, k: int = 3, verbose: bool = True) -> dict:
    """
    Vollständige Naive RAG-Pipeline:
    1. Retrieve: Relevante Chunks finden
    2. Augment: Prompt mit Kontext anreichern
    3. Generate: Prompt für LLM vorbereiten (hier: Ausgabe)
    """
    start = time.time()

    # Schritt 1: Retrieval
    kontext = retrieval(frage, index_sem, chunks_sem_alle, k=k)

    # Schritt 2: Prompt erstellen
    prompt = prompt_erstellen(frage, kontext)

    dauer = time.time() - start

    if verbose:
        print(f'❓ Frage: {frage}')
        print(f'⏱️  Retrieval-Zeit: {dauer*1000:.0f}ms')
        print(f'📎 {k} relevante Chunks gefunden:')
        for i, c in enumerate(kontext):
            print(f'   [{i+1}] {c["quelle"]} (Score: {c["score"]:.3f})')
        print(f'\n📝 Prompt ({len(prompt.split())} Wörter) bereit für LLM')
        print('\n--- Prompt (Ausschnitt) ---')
        print(prompt[:500] + '...')

    return {
        'frage': frage,
        'kontext': kontext,
        'prompt': prompt,
        'latenz_ms': dauer * 1000
    }


# Test
ergebnis = naive_rag("Brauche ich VPN für Homeoffice?")

## 🏋️ Übungen

### Übung 1 — Eigene Fragen testen

In [ ]:
# Deine eigene Frage hier eingeben:
meine_frage = "Wie lange dauert die Lohnfortzahlung bei Krankheit?"

ergebnis = naive_rag(meine_frage)

### Übung 2 — Eigenes Dokument hinzufügen

In [ ]:
# Füge ein eigenes Dokument hinzu und baue den Index neu auf:
neues_dokument = {
    "quelle": "handbuch_weiterbildung.txt",
    "inhalt": """
    Weiterbildung und Entwicklung

    Jeder Mitarbeiter hat Anspruch auf 5 Weiterbildungstage pro Jahr.
    Das Weiterbildungsbudget beträgt 1500 Euro jährlich.
    Anträge müssen bis Ende Februar für das laufende Jahr eingereicht werden.

    Für Online-Kurse (Coursera, Udemy, LinkedIn Learning) kann das Budget direkt
    über das HR-Portal beantragt werden. Bei Präsenzschulungen ist eine
    Vorabgenehmigung durch die Führungskraft erforderlich.
    """
}

# Dokumente + neues Dokument
dokumente_erweitert = dokumente + [neues_dokument]
chunks_erweitert = alle_chunks_erstellen(dokumente_erweitert, 'semantisch')

print(f'\n📚 Erweiterte Wissensbasis: {len(chunks_erweitert)} Chunks')

# Index neu aufbauen
index_erweitert, _ = index_aufbauen(chunks_erweitert)

# Test
ergebnis = naive_rag(
    "Wie viel Budget habe ich für Weiterbildungen?",
    # Überschreibe den globalen Index für diesen Test:
)
# Hinweis: Passe naive_rag() an, um den erweiterten Index zu nutzen

## 🔬 Übung 3 — Chunking-Größe experimentieren

In [ ]:
# Wie verändert sich die Qualität bei verschiedenen Chunk-Größen?
testfrage = "Was passiert mit Resturlaub am Jahresende?"

print('📊 Experiment: Verschiedene Chunk-Größen\n')
for groesse in [30, 80, 150]:
    chunks_exp = []
    for doc in dokumente:
        for c in chunking_fest(doc['inhalt'], chunk_groesse=groesse, ueberlappung=10):
            chunks_exp.append({'text': c, 'quelle': doc['quelle'], 'chunk_nr': 0})

    texte = [c['text'] for c in chunks_exp]
    embs = modell.encode(texte).astype('float32')
    idx_exp = faiss.IndexFlatL2(embs.shape[1])
    idx_exp.add(embs)

    ergebnisse = retrieval(testfrage, idx_exp, chunks_exp, k=1)
    print(f'Chunk-Größe {groesse:3d} Wörter → {len(chunks_exp):2d} Chunks')
    print(f'  Bester Treffer (Score: {ergebnisse[0]["score"]:.3f}):')
    print(f'  {ergebnisse[0]["text"][:150]}...')
    print()

## 🧩 Zusammenfassung — Session 2+3

| Thema | Kernpunkte |
|-------|------------|
| **Parse** | Dokumente einlesen, Text extrahieren (PDF → Text) |
| **Chunking fest** | Einfach, vorhersehbar, aber zerschneidet semantische Einheiten |
| **Chunking semantisch** | Respektiert Absatzgrenzen, bessere Qualität für strukturierte Texte |
| **Überlappung** | Verhindert Informationsverlust an Chunk-Grenzen |
| **Prompt-Konstruktion** | Kontext + Frage → präzises Instruction-Format für LLM |

### Was kommt in Session 4?

**Hybrides Retrieval**: Kombination von FAISS (semantisch) + BM25 (Keyword-Suche)
→ Besser als beide Methoden allein!

---

## 💡 Weiterführende Ressourcen

- 📄 [RAG Paper (Lewis et al., 2020)](https://arxiv.org/pdf/2005.11401)
- 🛠️ [LangChain Text Splitters Doku](https://python.langchain.com/docs/how_to/#text-splitters)
- 🔍 [FAISS GitHub](https://github.com/facebookresearch/faiss)
- 🤗 [sentence-transformers Modelle](https://huggingface.co/sentence-transformers)

---
*PIRAGE RAG Kurs — Session 2+3 | 🦃 [Copilotclaw/trainingdemo](https://github.com/Copilotclaw/trainingdemo)*